In [ ]:
using Bessels

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.7*2*pi
ω_b_0 = 7.05*2*pi

Lamb_shift = - 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.2
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37*2*pi


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.25
g2_max = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10*g2_max

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10; # 1/ns from Leon's paper
κ_2 = 4*g2_max^2/κ_b
κ_a = 3e-3*κ_2 #from Leon's paper



In [ ]:
t_max=20/κ_b

In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/simulator.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/visualization/plot_settings.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/visualization/plot_utils.jl");

In [ ]:
ϕ_a_0, q_a_0=0.0, 0.0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b);

In [ ]:
ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = simulator(t_max, system, drive, environment)

In [ ]:
config_params=(circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J = E_J, E_Ca=E_Ca, E_Cb=E_Cb, κ_a=0.0, κ_b=κ_b),
        drive_params=(ϵ_d = ϵ_d, ω_d = ω_b_0, buffer_phase = buffer_phase, ϵ_0 = ϵ_0, ω_p = 2ω_a-ω_b_0, pump_phase = pump_phase),
        init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0))

ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = simulator(t_max, ATS_CL_init, config_params, ATS_eq!)

In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/Code_Julia/Julia_file/classical_oscillators.jl");

In [ ]:
ϕ_CL_a_bis, ϕ_CL_b_bis, q_CL_a_bis, q_CL_b_bis, t_list_CL_bis = two_photon_CL(t_max;
        circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J = E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ_a=0.0,κ_b=κ_b,ϵ_d=ϵ_d,ω_d=ω_b_0),
        init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞)

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a, q_CL_a, t_list_CL, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
Re_a_CL = ϕ_CL_a_bis / sqrt(2*Z_a)
Im_a_CL = q_CL_a_bis * sqrt(Z_a/2)

ϕ_rotating_a =@. ϕ_CL_a_bis * cos(ω_a * t_list_CL) + q_CL_a_bis*Z_a* sin(-ω_a * t_list_CL)
q_rotating_a =@. -ϕ_CL_a_bis/Z_a * sin(-ω_a * t_list_CL) + q_CL_a_bis* cos(ω_a * t_list_CL)

Re_rotating_a_CL = ϕ_rotating_a / sqrt(2*Z_a)
Im_rotating_a_CL = q_rotating_a * sqrt(Z_a/2)

# ϕ_rotating_b =@. ϕ_CL_b * cos(ω_a * t_list) + q_CL_b*Z_a* sin(-ω_a * t_list)
# q_rotating_b =@. -ϕ_CL_b/Z_a * sin(-ω_a * t_list) + q_CL_b* cos(ω_a * t_list);
p1=plot(Re_a_CL,Im_a_CL)
p2=plot(Re_rotating_a_CL,Im_rotating_a_CL)


plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/baths.jl")
using MPSDynamics
n=3000
t_max=12/κ_b

bath=Exp_decay
bath_params =(Ω=30*2*pi, ω_cut_off=20*2*pi)

AB, J = bath(bath_params=bath_params,κ=κ_b, C_inv=C_b_inv)
chain_params = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=7000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

In [ ]:
ϕ_a_0, q_a_0=1.0, 0.0
ϕ_b_0, q_b_0=1.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModes(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
environment = OneChain(n, chain_params...)

In [ ]:
ϕ_a,ϕ_b,q_a,q_b,t_list = simulator(t_max, system, drive, environment)

In [ ]:
config_params_chain=(n, chain_params=chain_params, 
        circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J = E_J, E_Ca=E_Ca, E_Cb=E_Cb, κ_b=κ_b),
        drive_params=(ϵ_d = 0.0, ω_d = 0.0, buffer_phase = 0.0, ϵ_0 = 0.0, ω_p = 0.0, pump_phase = 0.0),
        init_mode_a=(ϕ_a_0=1.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=1.0,q_b_0 = 0.0))

In [ ]:
ϕ_a,ϕ_b,q_a,q_b,t_list = simulator(t_max, ATS_chain_init, config_params_chain, ATS_chain_eq!)

In [ ]:
plot(t_list*κ_b,ϕ_b)

In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

F,freqs=FFT_perso(ϕ_b, t_list)
f_shifted=abs(freqs[argmax(abs.(F))])
plot(freqs,abs.(F))
vline!([f_shifted], linestyle=:dash, color=:red, label="f_shifted")
plot!(xlims=(4,8))

In [ ]:
Lamb_shift = f_shifted - ω_b_0/(2*pi)

In [ ]:
ϕ_a_0, q_a_0=0.0, 0.0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = OneChain(n, chain_params...)

In [ ]:
ϕ_a,ϕ_b,q_a,q_b,t_list = simulator(t_max, system, drive, environment)

In [ ]:
config_params_chain=(n, chain_params=chain_params, 
        circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J = E_J, E_Ca=E_Ca, E_Cb=E_Cb, κ_b=κ_b),
        drive_params=(ϵ_d = ϵ_d, ω_d = ω_d, buffer_phase = buffer_phase, ϵ_0 = ϵ_0, ω_p = ω_p, pump_phase = pump_phase),
        init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0))

In [ ]:
ϕ_a,ϕ_b,q_a,q_b,t_list = simulator(t_max, ATS_chain_init, config_params_chain, ATS_chain_eq!)

In [ ]:
p1,p2 = plot_mode(ϕ_a, q_a, t_list, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
ϕ,ϕ_dot,t_list,Mq_chain,Mϕ_chain = simulator(t_max, system, drive, environment, solver=emission)

In [ ]:
A_chain = Mq_chain * Mϕ_chain

vals_chain, vecs_chain = eigen(A_chain)
ωk²_chain = real(vals_chain)
ωk_chain = sqrt.(ωk²_chain)
P_chain = real(vecs_chain)


Dq_chain = diag(P_chain'*(Mq_chain \ P_chain))  # uses linear solve instead of inv
Dϕ_chain = diag(P_chain' * Mϕ_chain * P_chain)

In [ ]:
# Solve P_chain * S = ϕ for S instead of inverting
S = P_chain \ ϕ
S_dot = P_chain \ ϕ_dot

# Energy in normal modes
Energies_normal_modes_chain = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2;


In [ ]:
f_a=ω_a/2pi
f_b=ω_b_0/2pi
f_p=abs(ω_p/2pi)
plot(ωk_chain/2pi,Energies_normal_modes_chain)
vline!([f_a],linestyle=:dash, color=:red)
annotate!((f_a)-.2,1e-0, Plots.text(L"\omega_d/2+\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a-2f_p],linestyle=:dash, color=:red)
annotate!((f_a-2f_p)-.2,1e-0, Plots.text(L"\omega_d/2-3\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a+2f_p],linestyle=:dash, color=:red)
annotate!((f_a+2f_p)-.2,1e-0, Plots.text(L"\omega_d/2+5\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a-4f_p],linestyle=:dash, color=:red)
annotate!((f_a-4f_p)-.2,1e-0, Plots.text(L"\omega_d/2-7\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a+4f_p],linestyle=:dash, color=:red)
annotate!((f_a+4f_p)-.2,1e-0, Plots.text(L"\omega_d/2+9\omega_p/2",18, :red, rotation = 90 ))

vline!([f_p],linestyle=:dash, color=:red)
annotate!(f_p-.2,1e-0, Plots.text(L"\omega_p", 18, :red, rotation = 90 ))
vline!([3f_p],linestyle=:dash, color=:red)
annotate!(3f_p-.2,1e-0, Plots.text(L"3\omega_p", 18, :red, rotation = 90 ))
vline!([f_b],linestyle=:dash, color=:red)

annotate!(f_b-.2,1e-8, Plots.text(L"\omega_d",18, :red, rotation = 90 ))
vline!([f_b-2f_p],linestyle=:dash, color=:red)
annotate!((f_b-2f_p)-.2,1e-8, Plots.text(L"\omega_d-2\omega_p",18, :red, rotation = 90 ))
vline!([f_b+2f_p],linestyle=:dash, color=:red)
annotate!((f_b+2f_p)-.2,1e-8, Plots.text(L"\omega_d+2\omega_p",18, :red, rotation = 90 ))
#vline!([f_b-4f_p],linestyle=:dash, color=:red)
#annotate!((f_b-4f_p)-.2,1e-8, Plots.text(L"\omega_d-4\omega_p",18, :red, rotation = 90 ))
vline!([f_b+4f_p],linestyle=:dash, color=:red)
annotate!((f_b+4f_p)-.2,1e-8, Plots.text(L"\omega_d+4\omega_p",18, :red, rotation = 90 ))

vline!([3f_b/2-5f_p/2],linestyle=:dash, color=:red)
annotate!((3f_b/2-5f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2-5\omega_p/2",18, :red, rotation = 90 ))
vline!([3f_b/2-f_p/2],linestyle=:dash, color=:red) 
annotate!((3f_b/2-f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2-\omega_p/2",18, :red, rotation = 90 ))
vline!([3f_b/2+3f_p/2],linestyle=:dash, color=:red)
annotate!((3f_b/2+3f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2+3\omega_p/2",18, :red, rotation = 90 ))
vline!([3f_b/2+7f_p/2],linestyle=:dash, color=:red)
annotate!((3f_b/2+7f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2+7\omega_p/2",18, :red, rotation = 90 ))

#vline!([2f_b-3f_p],linestyle=:dash, color=:red)
#annotate!((2f_b-3f_p)-.2,1e-8, Plots.text(L"2\omega_d-3\omega_p",18, :red, rotation = 90 ))
vline!([2f_b-f_p],linestyle=:dash, color=:red)
annotate!((2f_b-f_p)-.2,1e-8, Plots.text(L"2\omega_d-\omega_p",18, :red, rotation = 90 ))
vline!([2f_b+f_p],linestyle=:dash, color=:red)
annotate!((2f_b+f_p)-.2,1e-8, Plots.text(L"2\omega_d+\omega_p",18, :red, rotation = 90 ))

plot!(ylabel="Energy stored in the chain", xlabel="Frequency (GHz)", title = L"κt = "*"$(κ_b*t_max), "*L"n_{osc}="*"$(n)", legend=false, yaxis=:log, xlims=(0.1, 16), ylims=(1e-10, 1e2))
#savefig("../../Figures/Emission_spectrum/Energies_normal_modes_chain_two_photon_09_07.pdf")

In [ ]:
using ChaosTools

In [ ]:
n=3000

bath=Exp_decay
bath_params =(Ω=50*2*pi, ω_cut_off=20*2*pi)

AB, J = bath(bath_params=bath_params,κ=κ_b, C_inv=C_b_inv)
chain_params = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=7000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

In [ ]:
ω_d=ω_b_0+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d # Pump Frequency

In [ ]:
t_max = 10/κ_b

In [ ]:
ϕ_a_0, q_a_0=0.0, 0.0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

system_prepared = prepare(system)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
u_0_new = vcat(u_0,u_dot_0);

In [ ]:
ϵ_0_list = vcat(0.5:.05:1,1:.05:0.6pi)
ϵ_d_g2_ratio_list = 2:0.5:16

In [ ]:
using ProgressBars

In [ ]:
Lyap_coef_chaos = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_0_list))
drive = DriveTwoModes(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
drive_prepared = prepare(drive)
for i in ProgressBar(1:length(ϵ_0_list))
    ϵ_0_i = ϵ_0_list[i]
    drive.ϵ_0 = ϵ_0_i
    g2_max_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    for j in 1:length(ϵ_d_g2_ratio_list)
        ϵ_d_j = ϵ_d_g2_ratio_list[j]*g2_max_i
        drive.ϵ_d = ϵ_d_j
        drive_prepared.drive = drive
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        problem = CoupledODEs(equation_1ODE!, u_0_new, p)

        Lyap_coef_chaos[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = 100*dt)
    end
end